# 04 — Retention Levers & Segmentation

Two things in this notebook:

1. How much do `is_auto_renew`, `payment_plan_days`, and `payment_method_id` associate with
   retention — with selection effects flagged explicitly, since users who opt into auto-renew
   or an annual plan are likely a different population from those who don't, independent of
   whatever the lever itself does.
2. Which subscriber segments carry the most **revenue risk**, sized so recommendations can be
   prioritized by dollars, not just by rate.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats
import statsmodels.api as sm

sys.path.append(str(Path.cwd().parent))
from src import plotting

plotting.set_style()
periods = pd.read_parquet("../data/interim/periods.parquet")
members_cohort = pd.read_parquet("../data/interim/members_cohort.parquet")
resolved = periods[periods["outcome"] != "censored"]

## Lever 1: is_auto_renew

In [ ]:
auto_renew_summary = resolved.groupby("is_auto_renew")["renewed"].agg(["mean", "size"])
print(auto_renew_summary)

table = pd.crosstab(resolved["is_auto_renew"], resolved["renewed"])
chi2, p, dof, _ = stats.chi2_contingency(table)
print(f"\nchi2 = {chi2:.1f}, p = {p:.4g}")

**Selection effect:** users who opt into auto-renew are plausibly more committed / less price-sensitive to begin with. Auto-renew doesn't just mechanically remove the "forgot to renew" failure mode — it also correlates with unobserved intent to stay. This association is not evidence that *flipping* auto-renew on for a given user would retain them at this rate.

## Lever 2: payment_plan_days

In [ ]:
plan_days_summary = (
    resolved.groupby("payment_plan_days")["renewed"].agg(["mean", "size"])
    .query("size >= 500").sort_index()
)
print(plan_days_summary)

fig, ax = plotting.plot_segment_risk_bars(
    plan_days_summary.index.astype(str).tolist(),
    plan_days_summary["mean"].tolist(),
    "Renewal rate by plan length (plans with >=500 periods)",
    "Renewal rate",
    save_path="../figures/04_plan_days.png",
)

**Selection effect:** annual-plan subscribers already paid a large amount upfront and are a self-selected, likely lower-churn-propensity population — this is not evidence that moving a monthly subscriber onto an annual plan would produce the same retention rate for them.

## Lever 3: payment_method_id

In [ ]:
method_summary = (
    resolved.groupby("payment_method_id")["renewed"].agg(["mean", "size"])
    .query("size >= 500").sort_values("mean", ascending=False)
)
print("Highest-retention methods:")
print(method_summary.head(10))
print("\nLowest-retention methods:")
print(method_summary.tail(10))

Payment method carries selection too: some methods are disproportionately used by a particular subscriber type (e.g. auto-debit vs. convenience-store cash payment implies different banking access and possibly different income or commitment levels). Read these differences as descriptive segmentation, not as a lever to pull directly.

## Partial control: multivariate logistic regression
Puts all three levers in one model together with plan pricing, so each coefficient is "holding the others constant" rather than a raw marginal association. This partially addresses selection *between* the three levers, but not the deeper selection into any of them versus unobserved subscriber intent.

In [ ]:
top_methods = resolved["payment_method_id"].value_counts().head(8).index
reg_df = resolved[resolved["payment_method_id"].isin(top_methods)].copy()

X = pd.get_dummies(
    reg_df[["is_auto_renew", "payment_plan_days", "payment_method_id"]],
    columns=["payment_method_id"], drop_first=True,
)
X["plan_list_price_k"] = reg_df["plan_list_price"] / 1000.0
X = sm.add_constant(X.astype(float))
y = reg_df["renewed"].astype(int)

lever_model = sm.Logit(y, X).fit(disp=False)
print(lever_model.summary())

## Segmentation: where's the revenue risk concentrated?
Segment by plan length x auto-renew, sized by resolved-period revenue and lapse rate — this ranking is what should drive prioritization in the README's recommendation memo.

In [ ]:
resolved = resolved.copy()
resolved["plan_bucket"] = pd.cut(
    resolved["payment_plan_days"], bins=[0, 31, 100, 400], labels=["<=31d", "32-100d", "100d+"]
)
resolved["auto_renew_label"] = resolved["is_auto_renew"].map({0: "manual renew", 1: "auto-renew"})

segment = resolved.groupby(["plan_bucket", "auto_renew_label"], observed=True).agg(
    n_periods=("renewed", "size"),
    renewal_rate=("renewed", "mean"),
    revenue_at_stake=("actual_amount_paid", "sum"),
).reset_index()
segment["revenue_lost"] = segment["revenue_at_stake"] * (1 - segment["renewal_rate"])
segment = segment.sort_values("revenue_lost", ascending=False)
print(segment.to_string(index=False))

In [ ]:
labels = (segment["plan_bucket"].astype(str) + " / " + segment["auto_renew_label"]).tolist()
fig, ax = plotting.plot_segment_risk_bars(
    labels, segment["revenue_lost"].tolist(),
    "Revenue at risk by segment (unrenewed share x revenue)", "NT$ lost to non-renewal",
    save_path="../figures/04_segment_risk.png",
)

## Demographic cut: city and age band
(carrying forward the `bd` data-quality caveat from notebook 01 — invalid ages get their own bucket rather than being dropped or guessed at).

In [ ]:
demo = resolved.merge(
    members_cohort[["msno", "city", "bd", "gender"]], on="msno", how="left"
)
demo["age_valid"] = demo["bd"].between(10, 90)
demo["age_band"] = pd.cut(
    demo["bd"].where(demo["age_valid"]),
    bins=[10, 20, 30, 40, 50, 90], labels=["10-20", "20-30", "30-40", "40-50", "50+"],
)
demo["age_band"] = demo["age_band"].cat.add_categories(["unknown"]).fillna("unknown")

age_summary = demo.groupby("age_band", observed=True).agg(n=("renewed", "size"), renewal_rate=("renewed", "mean"))
print(age_summary)

top_cities = demo["city"].value_counts().head(10).index
city_summary = (
    demo[demo["city"].isin(top_cities)]
    .groupby("city").agg(n=("renewed", "size"), renewal_rate=("renewed", "mean"))
    .sort_values("renewal_rate")
)
print(city_summary)

## Section summary

- Rank segments by `revenue_lost`, not by renewal rate alone — a small segment with a terrible
  renewal rate can matter less in dollar terms than a large segment with a mediocre one. That
  ranking is what should drive prioritization in the recommendation memo.
- Every lever association above is confounded by selection into that lever; none of them should
  be reported as "switching X would retain Y% of subscribers" without a controlled experiment.